# 04d — LangGraph Advanced: Parallel Nodes & Subgraphs
**Domain:** IT / Complex Incident Response &nbsp;|&nbsp; **Time:** ~3 h  
**Key Concepts:** fan-out, parallel execution, subgraphs, map-reduce


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ ready")


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
from openai import OpenAI
import operator, datetime, random

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
print("✅ imports ok")


---
## 🔵 Example — Ex 04d-1: Sequential Fan-out to Diagnostic Nodes

In [ ]:
class DiagState(TypedDict):
    incident:       str
    cpu_report:     str
    memory_report:  str
    network_report: str
    summary:        str

def check_cpu(s: DiagState) -> dict:
    return {"cpu_report": f"CPU {random.randint(40,95)}% — {'ELEVATED' if random.random()>.5 else 'OK'}"}

def check_memory(s: DiagState) -> dict:
    free_gb = round(random.uniform(0.2, 8.0), 1)
    return {"memory_report": f"Free memory: {free_gb} GB — {'CRITICAL' if free_gb < 1 else 'OK'}"}

def check_network(s: DiagState) -> dict:
    ms = random.randint(5, 600)
    return {"network_report": f"Latency {ms} ms — {'ELEVATED' if ms > 200 else 'OK'}"}

def synthesise(s: DiagState) -> dict:
    resp = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user", "content":
            f"Summarise diagnostics in 2 sentences:\n{s['cpu_report']}\n{s['memory_report']}\n{s['network_report']}"}],
        max_tokens=120,
    )
    return {"summary": resp.choices[0].message.content}

dg = StateGraph(DiagState)
for name, fn in [("cpu", check_cpu), ("mem", check_memory),
                 ("net", check_network), ("synth", synthesise)]:
    dg.add_node(name, fn)
dg.set_entry_point("cpu")
dg.add_edge("cpu", "mem"); dg.add_edge("mem", "net"); dg.add_edge("net", "synth"); dg.add_edge("synth", END)
diag_app = dg.compile()

r = diag_app.invoke({"incident": "API timeout spike", "cpu_report": "",
                      "memory_report": "", "network_report": "", "summary": ""})
print(r["summary"])


---
## 🟡 Exercise — Ex 04d-2: Subgraph — Reusable Ticket Workflow

In [ ]:
show_exercise(
    "04d-2", "Subgraph: reusable ticket creation workflow",
    """Extract ticket creation into a standalone subgraph with nodes:
  validate_ticket → enrich_ticket → submit_ticket
Compile it and embed it as a node in two different parent graphs
(incident_parent and monitoring_parent).
run_ticket_subgraph(title, severity, desc) should work standalone.""",
    hints=[
        "Compile subgraph: sub_app = sub_graph.compile()",
        "Add as parent node: parent.add_node('ticket_flow', sub_app)",
        "Subgraph state keys can overlap with parent state",
    ],
    checks=[
        "ticket_subgraph is a compiled StateGraph",
        "run_ticket_subgraph() returns dict with 'ticket_id'",
        "Subgraph embedded in 2 parent graphs without errors",
    ],
)


In [ ]:
class TicketState(TypedDict):
    title:       str
    severity:    str
    description: str
    ticket_id:   str
    enriched:    bool

# TODO: define validate_ticket, enrich_ticket, submit_ticket nodes
# TODO: build and compile ticket_subgraph

ticket_subgraph = None   # ← replace with compiled graph

def run_ticket_subgraph(title: str, severity: str, desc: str) -> dict:
    # TODO: invoke ticket_subgraph and return final state
    pass

# TODO: define incident_parent_graph  (uses ticket_subgraph as a node)
# TODO: define monitoring_parent_graph (also uses ticket_subgraph)


In [ ]:
try:
    t = run_ticket_subgraph("API 500 errors", "high", "Endpoint returning 500 since deploy")
except Exception as e:
    t = None; print(f"Error: {e}")

check([
    (ticket_subgraph is not None,                    "ticket_subgraph compiled"),
    (t is not None,                                  "Subgraph runs standalone"),
    (isinstance(t, dict) if t else False,            "Returns dict"),
    ("ticket_id" in (t or {}),                       "'ticket_id' in result"),
], exercise_id="04d-2")


---
## 🔴 Challenge — Ex 04d-3: Map-Reduce with 5 Parallel Agents

In [ ]:
show_exercise(
    "04d-3", "Map-reduce: 5-service parallel incident analysis",
    """Given SERVICES list of 5 service names:
- Run 5 async diagnostic calls concurrently (asyncio.gather)
- Collect all findings into a list (reduce)
- Synthesise a root_cause hypothesis with one LLM call
Return: {findings: list[dict], root_cause: str}""",
    hints=[
        "async def diagnose_one(service) → call client inside async function",
        "import asyncio; findings = await asyncio.gather(*[diagnose_one(s) for s in SERVICES])",
        "Run in notebook: findings = asyncio.run(map_reduce_incident(SERVICES))",
    ],
    checks=[
        "findings list has exactly 5 entries",
        "'root_cause' key present in result",
        "All 5 services appear in findings",
    ],
)


In [ ]:
import asyncio

SERVICES = ["api-gateway", "auth-service", "database", "cache", "message-queue"]

async def diagnose_one(service: str) -> dict:
    """Async single-service diagnostic."""
    resp = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user",
                   "content": f"In one sentence, describe a plausible health issue for: {service}"}],
        max_tokens=60,
    )
    return {"service": service, "finding": resp.choices[0].message.content.strip()}

async def map_reduce_incident(services: list) -> dict:
    # TODO: run diagnose_one for all services concurrently
    # TODO: collect findings
    # TODO: one LLM call to synthesise root_cause
    pass

result_04d3 = asyncio.run(map_reduce_incident(SERVICES))
print(f"Services diagnosed: {len((result_04d3 or {}).get('findings', []))}")
print(f"Root cause: {(result_04d3 or {}).get('root_cause', '')[:200]}")

check([
    (result_04d3 is not None,                                      "map_reduce_incident() returns result"),
    (len((result_04d3 or {}).get("findings", [])) == 5,            "5 findings in result"),
    ("root_cause" in (result_04d3 or {}),                          "'root_cause' key present"),
], exercise_id="04d-3")


In [ ]:
check([
    (run_ticket_subgraph("t","low","d") is not None, "Ticket subgraph works (04d-2)"),
    (result_04d3 is not None,                        "Map-reduce works (04d-3)"),
    (len((result_04d3 or {}).get("findings",[])) == 5, "5 parallel findings"),
], exercise_id="04d-final")
progress()
